## A machine learning approach to volatility forecasting


###### In this notebook, I try to replicate the research by Christensen, Siggaard and Veliyev (2022). I use EURUSD exchange rate tick data from November 2025-January 2026, so with intraday trading data (bid and ask).

In [1]:
import numpy as np
import pandas as pd
from numpy.linalg import lstsq

###### First, we load in the exchange rate data, Compute 5-minute log-returns and daily realized variance.

In [2]:
def build_series(
    files=[
        "EURUSD_November2025.xlsx",
        "EURUSD_December2025.xlsx",
        "EURUSD_January2026.xlsx",
    ],
    rescale_to_spot=True,
    rescale_factor=1e5
):
    dfs=[]
    for f in files:
        df=pd.read_excel(f, engine="openpyxl", header=None)
        
        df.columns=["symbol", "timestamp", "bid", "ask"]

        df["datetime"]=pd.to_datetime(
            df["timestamp"],
            format="%Y%m%d %H:%M:%S.%f",
            errors="raise"
        )
        
        df=df[["symbol", "datetime", "bid", "ask"]]
        dfs.append(df)

    data=pd.concat(dfs, ignore_index=True)
    data=data.sort_values("datetime").set_index("datetime")

    if rescale_to_spot:
        data["bid"]=data["bid"]/rescale_factor
        data["ask"]=data["ask"]/rescale_factor

    data["mid"]=(data["bid"]+data["ask"])/2.0

    mid_5m=data["mid"].resample("5min").last().dropna()

    logret_5m=np.log(mid_5m / mid_5m.shift(1)).dropna()

    # Calculate daily realized variance
    rv_daily=logret_5m.pow(2).resample("1D").sum().dropna()
    rv_daily=rv_daily.replace(0, np.nan).dropna()
    rv_daily=rv_daily[rv_daily > 0]
    rv_daily=rv_daily.sort_index()

    return data, mid_5m, logret_5m, rv_daily

###### Next, the exogenous variables are loaded in, which will be used in the HAR-X and L-HAR-X regressions

In [8]:
data, mid_5m, logret_5m, rv_daily = build_series()
rv  = rv_daily.copy().sort_index()
idx = rv.index

#Load exogenous variables
def load_XVAR_excel(file: str, date_col: str = "observation_date") -> pd.Series:
    df = pd.read_excel(file, engine="openpyxl", decimal=",")
    if date_col not in df.columns:
        raise ValueError(f"{file}: expected a '{date_col}' column")
    value_cols = [c for c in df.columns if c != date_col]
    if not value_cols:
        raise ValueError(f"{file}: no value column besides '{date_col}'")
    value_col = value_cols[0]

    df[date_col]  = pd.to_datetime(df[date_col], errors="coerce")
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")

    series = (
        df.dropna(subset=[date_col, value_col])
          .sort_values(date_col)
          .drop_duplicates(subset=[date_col], keep="last")
          .set_index(date_col)[value_col]
          .rename(value_col)
    )

    return series

def align_to_idx(series):
    return series.reindex(idx).ffill()

EPU     = align_to_idx(load_XVAR_excel("EPUIndexUS.xlsx"))
FEDFUN  = align_to_idx(load_XVAR_excel("FedfundsrateUS.xlsx"))
INFL    = align_to_idx(load_XVAR_excel("InflationUS.xlsx"))
NASDAQ  = align_to_idx(load_XVAR_excel("NasdaqcompositeUS.xlsx"))
SP500   = align_to_idx(load_XVAR_excel("SP500US.xlsx"))
T10Y    = align_to_idx(load_XVAR_excel("TenyearTrateUS.xlsx"))
VIX     = align_to_idx(load_XVAR_excel("VIXUS.xlsx"))

###### Construct one large dataset with all the variables needed in this replication

In [9]:
# HAR components
rv_lag1  = rv.shift(1)
rv_week  = rv_lag1.rolling(5).mean()
rv_month = rv_lag1.rolling(22).mean()

# Log-HAR components
log_rv       = np.log(rv)
log_rv_lag1  = log_rv.shift(1)
log_rv_week  = log_rv_lag1.rolling(5).mean()
log_rv_month = log_rv_lag1.rolling(22).mean()

# Quarticity HARQ
n_per_day = int(logret_5m.groupby(logret_5m.index.date).size().median())
rq = ((n_per_day / 3) * logret_5m.pow(4).resample("1D").sum()).reindex(idx)
sqrt_rq_lag1 = rq.shift(1).pow(0.5)
interaction = sqrt_rq_lag1 * rv_lag1

# SHAR semivariances
pos_sq = (logret_5m.where(logret_5m > 0, 0))**2
neg_sq = (logret_5m.where(logret_5m < 0, 0))**2

rv_pos = pos_sq.resample("1D").sum().reindex(idx)
rv_neg = neg_sq.resample("1D").sum().reindex(idx)

rvpos_lag1 = rv_pos.shift(1)
rvneg_lag1 = rv_neg.shift(1)

# LevHAR
daily_ret = logret_5m.resample("1D").sum().reindex(idx)
ret_lag1   = daily_ret.shift(1)
ret_week   = ret_lag1.rolling(5).mean()
ret_month  = ret_lag1.rolling(22).mean()

retneg_lag1  = np.minimum(ret_lag1, 0)
retneg_week  = np.minimum(ret_week, 0)
retneg_month = np.minimum(ret_month, 0)

#HAR-X and its lags
exovars = {
    "VIX": VIX,
    "T10Y": T10Y,
    "SP500": SP500,
    "NASDAQ": NASDAQ,
    "INFL": INFL,
    "FEDFUN": FEDFUN,
    "EPU": EPU,
}

exolags = {}
for name, series in exovars.items():
    lag1   = series.shift(1)
    week   = lag1.rolling(5).mean()
    month  = lag1.rolling(22).mean()

    exolags[f"{name}_lag1"]   = lag1
    exolags[f"{name}_week"]   = week
    exolags[f"{name}_month"]  = month
exo_df = pd.DataFrame(exolags)

df= pd.DataFrame({

    "rv": rv,
    "logrv": log_rv,

    # HAR
    "rv_lag1": rv_lag1,
    "rv_week": rv_week,
    "rv_month": rv_month,

    # HARQ
    "sqrt_rq_lag1": sqrt_rq_lag1,
    "interaction": interaction,

    # SHAR
    "rvpos_lag1": rvpos_lag1,
    "rvneg_lag1": rvneg_lag1,

    # LevHAR
    "retneg_lag1": retneg_lag1,
    "retneg_week": retneg_week,
    "retneg_month": retneg_month,

    # Log-HAR
    "logrv_lag1": log_rv_lag1,
    "logrv_week": log_rv_week,
    "logrv_month": log_rv_month,
})

df = pd.concat([df, exo_df], axis=1)
df = df.dropna()
df.head()

,rv,logrv,rv_lag1,rv_week,rv_month,sqrt_rq_lag1,interaction,rvpos_lag1,rvneg_lag1,retneg_lag1,...,NASDAQ_month,INFL_lag1,INFL_week,INFL_month,FEDFUN_lag1,FEDFUN_week,FEDFUN_month,EPU_lag1,EPU_week,EPU_month
datetime,,,,,,,,,,,,,,,,,,,,,
2025-11-28,36.392670,3.594367,51.909253,31.891785,54.934580,109.143728,5665.569374,25.955405,25.953848,-0.000194,...,22976.183636,2.23,2.230,2.267273,3.88,3.880,3.875909,334.67,288.738,336.151364
2025-12-01,58.316021,4.065877,36.392670,37.823658,49.088629,79.470859,2892.156747,18.144995,18.247675,-0.000379,...,22954.864091,2.23,2.228,2.263636,3.89,3.882,3.876818,323.26,282.472,350.600000
2025-12-02,33.857335,3.522156,58.316021,40.022042,49.580851,123.912566,7226.087821,28.252015,30.064006,0.000000,...,22951.558636,2.24,2.230,2.261364,3.89,3.884,3.877727,257.88,294.228,349.335455
2025-12-03,59.751261,4.090190,33.857335,41.714502,45.152710,79.017083,2675.307831,16.929158,16.928177,0.000000,...,22947.643636,2.24,2.234,2.258636,3.89,3.886,3.878636,292.27,287.934,348.947273
2025-12-04,74.745669,4.314091,59.751261,48.045308,45.088280,123.992872,7408.730498,30.895075,28.856186,0.000000,...,22965.830000,2.24,2.236,2.256818,3.89,3.888,3.879545,357.52,313.120,350.800909


###### Check the stationarity of all regressors and dep. variable in HAR-X regression, as well as the multicollinearity

In [10]:
from statsmodels.tsa.stattools import adfuller

variables = [
    "rv_lag1", "rv_week", "rv_month",
    "VIX_lag1", "T10Y_lag1", "SP500_lag1", "NASDAQ_lag1",
    "INFL_lag1", "FEDFUN_lag1", "EPU_lag1",
    "VIX_week", "T10Y_week", "SP500_week", "NASDAQ_week",
    "INFL_week", "FEDFUN_week", "EPU_week",
    "VIX_month", "T10Y_month", "SP500_month", "NASDAQ_month",
    "INFL_month", "FEDFUN_month", "EPU_month"]
adf_results = {}
for var in variables:
    series = df[var].dropna()
    result = adfuller(series, autolag="AIC")
    adf_results[var] = {
        "ADF Statistic": result[0],
        "p-value": result[1]   
    }
adf_df = pd.DataFrame(adf_results).T
print(adf_df)

corr_matrix = df[variables].corr()
print(corr_matrix)

              ADF Statistic       p-value
rv_lag1           -2.112971  2.393312e-01
rv_week           -2.263175  1.841198e-01
rv_month          -3.358787  1.244668e-02
VIX_lag1          -2.344849  1.579107e-01
T10Y_lag1         -2.237947  1.927623e-01
SP500_lag1        -2.313085  1.677824e-01
NASDAQ_lag1       -4.562096  1.516231e-04
INFL_lag1         -0.222563  9.357782e-01
FEDFUN_lag1       -1.934408  3.160515e-01
EPU_lag1          -6.760872  2.794441e-09
VIX_week          -1.066589  7.282103e-01
T10Y_week         -2.626199  8.770239e-02
SP500_week        -1.067147  7.279940e-01
NASDAQ_week       -2.983353  3.647856e-02
INFL_week         -0.332537  9.207680e-01
FEDFUN_week       -5.385165  3.638889e-06
EPU_week          -4.450655  2.412043e-04
VIX_month         -2.461472  1.251322e-01
T10Y_month        -2.331147  1.621190e-01
SP500_month       -1.112723  7.099715e-01
NASDAQ_month      -1.231749  6.597912e-01
INFL_month         0.311045  9.778456e-01
FEDFUN_month      -2.057978  2.617

c:\Users\gnijland\Documents\Code_Thesis_AEF\AEF-Thesis\.venv\Lib\site-packages\statsmodels\regression\linear_model.py:955: RuntimeWarning: divide by zero encountered in log
  llf = -nobs2*np.log(2*np.pi) - nobs2*np.log(ssr / nobs) - nobs2


###### Because SP500 and NASDAQ have high correlation, an auxillary regressions is performed, and the residuals of this regression are used instead of both variables in the HAR-X and L-HAR-X regressions

In [12]:
import statsmodels.api as sm
nas = df["NASDAQ_lag1"]
sp  = df["SP500_lag1"]

aux_df = pd.concat([nas, sp], axis=1).dropna()
aux_df.columns = ["NASDAQ_lag1", "SP500_lag1"]

X = sm.add_constant(aux_df["SP500_lag1"])
y = aux_df["NASDAQ_lag1"]

aux_model = sm.OLS(y, X).fit()

resid_lag1 = aux_model.resid
resid_lag1.name = "resid_lag1"

df["resid_lag1"] = resid_lag1

###### Check if error term contains a unit root

In [13]:
X = df[variables]
y = df["rv"]
Xc = sm.add_constant(X)
ols_model = sm.OLS(y, Xc).fit()
resid = ols_model.resid
adf_stat, p_value, lags, nobs, crit_vals, icbest = adfuller(resid.dropna(), autolag='AIC')

print("\n=== ADF Test on Regression Residuals ===")
print(f"ADF Statistic: {adf_stat:.4f}")
print(f"p-value:       {p_value:.4f}")
print(f"Used Lags:     {lags}")
print(f"N Obs:         {nobs}")
print("Critical Values:")
for k, v in crit_vals.items():
    print(f"   {k}: {v:.4f}")

if p_value < 0.05:
    print("\nResult: Residuals are STATIONARY (reject unit root).")
else:
    print("\nResult: Residuals are NON‑stationary (cannot reject unit root).")


=== ADF Test on Regression Residuals ===
ADF Statistic: -4.2496
p-value:       0.0005
Used Lags:     8
N Obs:         45
Critical Values:
   1%: -3.5848
   5%: -2.9283
   10%: -2.6023

Result: Residuals are STATIONARY (reject unit root).


In [14]:
#Create test and training datasets
split_idx = int(0.8 * len(df))
train = df.iloc[:split_idx]
test  = df.iloc[split_idx:]

train_idx = train.index
test_idx  = test.index

###### Create rolling window forecast function and function to calculate mean-squared error

In [15]:
def rolling_forecast(train_df, test_df, y_col, feature_cols):
    preds = []
    actuals = []

    y_train_series = train_df[y_col]
    X_train_block  = train_df[feature_cols]

    for i, t in enumerate(test_df.index):
        if i == 0:
            X_in = X_train_block
            y_in = y_train_series
        else:
            X_in = pd.concat([X_train_block, test_df[feature_cols].iloc[:i]])
            y_in = pd.concat([y_train_series, test_df[y_col].iloc[:i]])

        Xc = sm.add_constant(X_in, has_constant='add')
        res = sm.OLS(y_in, Xc).fit()

        X_t = sm.add_constant(test_df[feature_cols].loc[[t]], has_constant='add')
        pred = res.predict(X_t).iloc[0]

        preds.append(pred)
        actuals.append(test_df[y_col].loc[t])

    return pd.Series(preds, index=test_df.index), pd.Series(actuals, index=test_df.index)

def mse(pred, actual):
    return ((pred - actual)** 2).mean()

In [16]:
# HAR
har_fc, har_ac = rolling_forecast(
    train, test,
    y_col="rv",
    feature_cols=["rv_lag1","rv_week","rv_month"]
)
print("HAR MSE:", mse(har_fc, har_ac))

# HARQ
harq_fc, harq_ac = rolling_forecast(
    train, test,
    y_col="rv",
    feature_cols=["rv_lag1","interaction","rv_week","rv_month"]
)
print("HARQ MSE:", mse(harq_fc, harq_ac))

# Log-HAR
log_fc_log, log_ac_log = rolling_forecast(
    train, test,
    y_col="logrv",
    feature_cols=["logrv_lag1","logrv_week","logrv_month"]
)

# Jensen correction
resid_var = (log_ac_log - log_fc_log).var()
log_fc = np.exp(log_fc_log + 0.5*resid_var)
log_ac = np.exp(log_ac_log)
print("LogHAR MSE:", mse(log_fc, log_ac))

# LevHAR
levhar_fc, levhar_ac = rolling_forecast(
    train, test,
    y_col="rv",
    feature_cols=["rv_lag1","rv_week","rv_month",
                  "retneg_lag1","retneg_week","retneg_month"]
)
print("LevHAR MSE:", mse(levhar_fc, levhar_ac))

# SHAR
shar_fc, shar_ac = rolling_forecast(
    train, test,
    y_col="rv",
    feature_cols=["rvneg_lag1","rvpos_lag1","rv_week","rv_month"]
)
print("SHAR MSE:", mse(shar_fc, shar_ac))

HAR MSE: 357.8044757085845
HARQ MSE: 366.7005632234568
LogHAR MSE: 542.3830342038295
LevHAR MSE: 513.1644735418215
SHAR MSE: 371.42689273445313


In [ ]:
print("\n HAR-X forecasts MSE")
harx_fc, harx_actual=rolling_forecast(
        train, test, y_col="rv",
        feature_cols=variables
    )
print(mse(harx_fc, harx_actual))

endogenous_lags = ["rv_lag1", "rv_week", "rv_month"]
exogenous_lags = [v for v in variables if v not in endogenous_lags]
                  
# L-HAR-X forecasts and MSE
print("\n L-HAR-X forecasts MSE")
lharx_fc_log, lharx_actual_log = rolling_forecast(
    train, test,
    feature_cols=["logrv_lag1", "logrv_week", "logrv_month"]+exogenous_lags,
    y_col="logrv"
    )
# Jensen correction 
resid_var=(lharx_actual_log - lharx_fc_log).var()
lharx_fc=np.exp(lharx_fc_log + 0.5 * resid_var)
lharx_actual=np.exp(lharx_actual_log)

print(mse(lharx_fc, lharx_actual))

###### Now, apply the following regularization techniques to the HAR-X regression; Ridge regression, LASSO, Elastic Net, Post LASSO, Adaptive LASSO

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet

def to_scalar(x):
    if hasattr(x, "values"):
        x = x.values
    return float(np.asarray(x).reshape(-1)[0])

def scalar_pred(model, X):
    yhat = model.predict(X)
    arr = np.asarray(yhat).ravel()
    return float(arr[0])

# Hyperparameter tuning for Ridge/Lasso/Elastic Net
def tune_regularization(model_class, X_train, y_train, X_val, y_val):
    lambdas = np.logspace(-5, 2, 80)
    alphas  = np.linspace(0, 1, 8)

    best_mse = np.inf
    best_model = None
    best_scaler = None

    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X_train.values)
    Xvl = scaler.transform(X_val.values)

    for lam in lambdas:
        if model_class == ElasticNet:
            for a in alphas:
                model = ElasticNet(alpha=lam, l1_ratio=a, max_iter=10000)
                model.fit(Xtr, y_train)
                mse_val = ((model.predict(Xvl) - y_val)**2).mean()
                if mse_val < best_mse:
                    best_mse = mse_val
                    best_model = model
                    best_scaler = scaler

        else:
            model = model_class(alpha=lam, max_iter=10000)
            model.fit(Xtr, y_train)
            mse_val = ((model.predict(Xvl) - y_val)**2).mean()
            if mse_val < best_mse:
                best_mse = mse_val
                best_model = model
                best_scaler = scaler

    return best_model, best_scaler

# Rolling window (train + validation) for Ridge/Lasso/Elastic Net
def rolling_regularized(model_class, X, y, train_size=0.7, val_size=0.1):
    T = len(X)
    train_end = int(train_size * T)
    val_end   = int((train_size + val_size) * T)
    preds, acts = [], []
    for t in range(val_end, T):

        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_val   = X.iloc[train_end:val_end]
        y_val   = y.iloc[train_end:val_end]

        X_test  = X.iloc[t:t+1]

        best_model, scaler = tune_regularization(model_class, X_train, y_train, X_val, y_val)

        pred = scalar_pred(best_model, scaler.transform(X_test.values))
        preds.append(pred)
        acts.append(y.iloc[t])

        train_end += 1
        val_end   += 1

    idx = X.index[int((train_size + val_size) * T):]
    return pd.Series(preds, index=idx), pd.Series(acts, index=idx)

# Adaptive Lasso
def rolling_adaptive_lasso(X, y, train_size=0.7, val_size=0.1):
    T = len(X)
    train_end = int(train_size * T)
    val_end   = int((train_size + val_size)*T)
    preds, acts = [], []
    for t in range(val_end, T):

        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_val   = X.iloc[train_end:val_end]
        y_val   = y.iloc[train_end:val_end]

        X_test  = X.iloc[t:t+1]

        Xc = sm.add_constant(X_train)
        beta = sm.OLS(y_train, Xc).fit().params[1:]   
        weights = 1 / (np.abs(beta) + 1e-6)

        Xw_train = X_train * weights.values
        Xw_val   = X_val   * weights.values
        Xw_test  = X_test  * weights.values

        best_lasso, scaler = tune_regularization(Lasso, Xw_train, y_train, Xw_val, y_val)
        pred = scalar_pred(best_lasso, scaler.transform(Xw_test.values))

        preds.append(pred)
        acts.append(y.iloc[t])

        train_end += 1
        val_end   += 1

    idx = X.index[int((train_size + val_size) * T):]
    return pd.Series(preds, index=idx), pd.Series(acts, index=idx)

# Post-Lasso
def rolling_post_lasso(X, y, train_size=0.7, val_size=0.1):

    T = len(X)
    train_end = int(train_size * T)
    val_end   = int((train_size + val_size) * T)

    preds, acts = [], []

    for t in range(val_end, T):

        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_val   = X.iloc[train_end:val_end]
        y_val   = y.iloc[train_end:val_end]

        X_test  = X.iloc[t:t+1]

        # Stage 1: tuned Lasso
        best_lasso, scaler = tune_regularization(Lasso, X_train, y_train, X_val, y_val)
        mask = best_lasso.coef_ != 0

        if mask.sum() == 0:
            pred = float(y_train.mean())

        else:
            Xs_train = X_train.iloc[:, mask].values
            Xs_test  = X_test.iloc[:, mask].values

            X_train_mat = np.column_stack([np.ones(len(Xs_train)), Xs_train])
            X_test_mat  = np.column_stack([np.ones(len(Xs_test)),  Xs_test])

            beta, *_ = np.linalg.lstsq(X_train_mat, y_train.values, rcond=None)
            pred = to_scalar(X_test_mat @ beta)  

        preds.append(pred)
        acts.append(y.iloc[t])

        train_end += 1
        val_end   += 1

    idx = X.index[int((train_size + val_size) * T):]
    return pd.Series(preds, index=idx), pd.Series(acts, index=idx)

###### Finally, let's check forecasting performance

In [ ]:
X = df[variables]
y = df["rv"]

# Regularized HAR-X Models
print("\nRIDGE HAR-X")
rr_fc, rr_act = rolling_regularized(Ridge, X, y)
print("MSE:", mse(rr_fc, rr_act))

print("\nLASSO HAR-X")
la_fc, la_act = rolling_regularized(Lasso, X, y)
print("MSE:", mse(la_fc, la_act))

print("\nELASTIC NET HAR-X")
en_fc, en_act = rolling_regularized(ElasticNet, X, y)
print("MSE:", mse(en_fc, en_act))

print("\nADAPTIVE LASSO HAR-X")
ala_fc, ala_act = rolling_adaptive_lasso(X, y)
print("MSE:", mse(ala_fc, ala_act))

print("\nPOST-LASSO HAR-X")
pl_fc, pl_act = rolling_post_lasso(X, y)
print("MSE:", mse(pl_fc, pl_act))

###### Now, we move to the ML models; Regression tree methods (Bootstrap aggregation, Random Forest, Gradient Boosting) and thereafter, Neural Networks

In [ ]:
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor

def rolling_ml(model, X, y, train_size=0.7, val_size=0.1):

    T = len(X)
    train_end = int(train_size * T)
    val_end   = int((train_size + val_size) * T)

    preds = []
    acts  = []

    for t in range(val_end, T):

        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_val   = X.iloc[train_end:val_end]
        y_val   = y.iloc[train_end:val_end]

        X_test  = X.iloc[t:t+1]
        y_test  = y.iloc[t]

        # standardize train/val/test using training only
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X_train.values)
        Xvl = scaler.transform(X_val.values)
        Xte = scaler.transform(X_test.values)

        # fit model (no tuning inside)
        model.fit(Xtr, y_train)

        # predict
        pred = model.predict(Xte)[0]
        preds.append(pred)
        acts.append(y_test)

        # expand window
        train_end += 1
        val_end   += 1

    idx = X.index[int((train_size + val_size)*T):]
    return pd.Series(preds, index=idx), pd.Series(acts, index=idx)

bag_model = BaggingRegressor(
    estimator=DecisionTreeRegressor(min_samples_leaf=5),
    n_estimators=500,
    bootstrap=True,
    n_jobs=-1,
    random_state=0
)

bag_fc, bag_act = rolling_ml(bag_model, X, y)
print("Bagging MSE:", mse(bag_fc, bag_act))

mtry = max(1, J // 3)

rf_model = RandomForestRegressor(
    n_estimators=500,
    min_samples_leaf=5,
    max_features=mtry,
    bootstrap=True,
    n_jobs=-1,
    random_state=0
)

rf_fc, rf_act = rolling_ml(rf_model, X, y)
print("Random Forest MSE:", mse(rf_fc, rf_act))

gb_depths = [1, 2]
gb_lrs    = [0.01, 0.1]
gb_trees  = list(range(50, 501, 50))

def tune_gb(X_train, y_train, X_val, y_val):

    best_mse = np.inf
    best_model = None
    scaler = StandardScaler()

    Xtr = scaler.fit_transform(X_train.values)
    Xvl = scaler.transform(X_val.values)

    for depth in gb_depths:
        for lr in gb_lrs:
            for n_trees in gb_trees:

                model = GradientBoostingRegressor(
                    max_depth=depth,
                    learning_rate=lr,
                    n_estimators=n_trees,
                    subsample=1.0,
                    random_state=0
                )

                model.fit(Xtr, y_train)
                mse_val = ((model.predict(Xvl) - y_val)**2).mean()

                if mse_val < best_mse:
                    best_mse = mse_val
                    best_model = model
                    best_scaler = scaler

    return best_model, best_scaler

def rolling_gb(X, y, train_size=0.7, val_size=0.1):

    T = len(X)
    train_end = int(train_size * T)
    val_end   = int((train_size + val_size)*T)

    preds = []
    acts  = []

    for t in range(val_end, T):

        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_val   = X.iloc[train_end:val_end]
        y_val   = y.iloc[train_end:val_end]

        X_test  = X.iloc[t:t+1]
        y_test  = y.iloc[t]

        best_gb, scaler = tune_gb(X_train, y_train, X_val, y_val)

        Xte = scaler.transform(X_test.values)
        pred = best_gb.predict(Xte)[0]

        preds.append(pred)
        acts.append(y_test)

        train_end += 1
        val_end   += 1

    idx = X.index[int((train_size + val_size)*T):]
    return pd.Series(preds, index=idx), pd.Series(acts, index=idx)

gb_fc, gb_act = rolling_gb(X, y)
print("Gradient Boosting MSE:", mse(gb_fc, gb_act))

Bagging MSE: 353.0282392167068
Random Forest MSE: 419.11749287102083
Gradient Boosting MSE: 265.4777235644552


In [23]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LeakyReLU
from tensorflow.keras.initializers import GlorotNormal
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping


# ============================================================
# 1. Neural Network Builder (Paper defaults)
# ============================================================
def build_nn(layers, input_dim):
    """
    Build a feedforward NN with:
    - LeakyReLU (alpha=0.01)
    - GlorotNormal initializer
    - Dropout (0.8)
    - Adam optimizer (learning_rate=0.001)
    """

    model = Sequential()

    # First layer
    model.add(Dense(layers[0], kernel_initializer=GlorotNormal(), input_dim=input_dim))
    model.add(LeakyReLU(alpha=0.01))
    model.add(Dropout(0.8))

    # Additional hidden layers
    for units in layers[1:]:
        model.add(Dense(units, kernel_initializer=GlorotNormal()))
        model.add(LeakyReLU(alpha=0.01))
        model.add(Dropout(0.8))

    # Output layer (regression)
    model.add(Dense(1, activation="linear"))

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="mse",
    )

    return model


# ============================================================
# 2. NN architectures NN1–NN4 (geometric pyramid)
# ============================================================
NN_ARCHS = {
    "NN1": [2],
    "NN2": [4, 2],
    "NN3": [8, 4, 2],
    "NN4": [16, 8, 4, 2]
}


# ============================================================
# 3. Train an ensemble of 100 networks ONCE (fixed-window)
#    Select best K networks with lowest training loss
# ============================================================
def train_nn_ensemble(model_id, X_train, y_train, ensemble_size=10):

    input_dim = X_train.shape[1]
    arch = NN_ARCHS[model_id]

    models = []
    losses = []

    for _ in range(100):   # Train 100 NNs (paper)
        model = build_nn(arch, input_dim)

        early = EarlyStopping(
            monitor="loss",
            patience=100,
            restore_best_weights=True,
            verbose=0
        )

        model.fit(
            X_train, y_train,
            epochs=500,       # paper default
            batch_size=32,
            verbose=0,
            callbacks=[early]
        )

        # Evaluate training loss
        loss = model.evaluate(X_train, y_train, verbose=0)
        losses.append(loss)
        models.append(model)

    # Pick the best K models
    best_idx = np.argsort(losses)[:ensemble_size]
    best_models = [models[i] for i in best_idx]

    return best_models


# ============================================================
# 4. Forecast full out-of-sample using FIXED trained NNs
# ============================================================
def forecast_nn_fixed(models, scaler, X_test):
    preds = []

    for t in range(len(X_test)):
        x = scaler.transform(X_test.iloc[t:t+1].values)
        # Average prediction of all ensemble models
        pred = np.mean([m.predict(x)[0, 0] for m in models])
        preds.append(pred)

    return pd.Series(preds, index=X_test.index)


# ============================================================
# 5. Complete NN pipeline: train once, forecast full test set
# ============================================================
def run_nn(model_id, train_df, test_df, y_col, feature_cols, ensemble_size=10):

    X_train = train_df[feature_cols]
    y_train = train_df[y_col]

    X_test  = test_df[feature_cols]
    y_test  = test_df[y_col]

    # Standardize features using training window only
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X_train.values)

    # Train ensemble ONCE
    models = train_nn_ensemble(model_id, Xtr, y_train.values, ensemble_size)

    # Forecast the whole out-of-sample period
    preds = forecast_nn_fixed(models, scaler, X_test)

    return preds, y_test

# ============================================================
# RUN ALL EIGHT NN MODELS (NN1–NN4 with ensembles 1 and 10)
# ============================================================

nn_results = {}   # store MSEs

for model_id in ["NN1", "NN2", "NN3", "NN4"]:
    for K in [1, 10]:

        print(f"\nRunning {model_id} with ensemble size {K} …")

        nn_fc, nn_act = run_nn(
            model_id=model_id,
            train_df=train,
            test_df=test,
            y_col="rv",
            feature_cols=variables,
            ensemble_size=K
        )

        mse_val = ((nn_fc - nn_act)**2).mean()

        key = f"{model_id}_{K}"
        nn_results[key] = mse_val

        print(f"{key} MSE: {mse_val:.6f}")



Running NN1 with ensemble size 1 …


c:\Users\gnijland\Documents\Code_Thesis_AEF\AEF-Thesis\.venv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\gnijland\Documents\Code_Thesis_AEF\AEF-Thesis\.venv\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


KeyboardInterrupt: 